# NetraSetu - Model 1 v2: DR Severity Classifier (Branch A), retrain

**Built, not executed here.** This notebook was prepared on a dev machine with
no Kaggle access, no EyePACS data, and 16GB free disk (EyePACS alone is
80GB+) -- see the v2 training-run report for the full explanation. Run it on
Kaggle (T4/P100, EyePACS attachable as a dataset without downloading it to
your own disk), the same way `train_classifier_kaggle.ipynb` (v1) was
actually trained.

**What changed from v1, and why (v2 task items 1-5, 7):**

| # | Change | Where |
|---|---|---|
| 1 | Input resolution 384 -> **512** | `IMG_SIZE` below |
| 2 | + EyePACS (curated subset) + a second, binary referable/non-referable head | data loading, `DRClassifierV2` |
| 3 | Ordinal loss's penalty made ASYMMETRIC: under-grading costs `(true-pred)^2`, over-grading costs half that | `OrdinalWeightedCEv2` |
| 4 | Class weights: grade 4 explicitly boosted above grade 3 (v1 had this backwards: 2.34 vs 2.92, purely because grade 3 has fewer raw examples than grade 4 -- inverse-frequency weighting alone cannot know grade 4 is clinically worse to miss) | class-weight cell |
| 5 | Binary head: focal loss + EyePACS referable oversampling + a validation-ROC threshold sweep locked at >=90% sensitivity | binary-head cells |
| 7 | Evaluation reports grade-4 recall, grade-1 recall, referable sens/spec, AND QWK on the RECOVERED held-out split -- all four, every run | evaluation cell |

**Item 6** (M5 microaneurysm count -> grade-0-vs-1 decision) is NOT in this
notebook -- it's inference-time rule-engine logic, not a training change. It
was investigated with real inference on real held-out images (see
`experiments/investigateM5Grade1.py`) and NOT implemented as an automatic
grade override: a naive count threshold would have fixed at most 3/4 real
grade-1 misses while mis-escalating roughly half of a small true-grade-0
sample tested. What ships instead: `gradingOrchestrator.js`'s existing
branch-disagreement mechanism already routes this exact case (CNN=0,
rule-engine>=1) to mandatory human review, now labeled distinctly
(`grade0Vs1Disagreement`) for monitoring.

**Item 8 is NOT optional and is NOT in this notebook either** -- it happens
AFTER this notebook produces a finished, evaluated checkpoint, as a separate
MATLAB step. See this notebook's last cell and the v2 report's closing
section. `calibration_v1.json`'s `qhat=0.8432` was fitted for v1 at 384x384
and a version guard (`branchAInfer.py` / `branchAInferMatlab.m`) will now
REFUSE to apply it to a v2 model automatically -- but that guard only helps
if v2's calibration is actually re-fitted and the file actually overwritten.
It is not.

## Mandatory pre-flight: exclude the recovered held-out set

`models/Model1/branchA_v1_{test,val}_{ids,labels}.npy` are v1's OWN saved
split arrays -- the only trustworthy record of which images were genuinely
held out, because they were written at split time, before any model existed
to leak into them. Upload them as a small private Kaggle dataset alongside
APTOS/IDRiD/EyePACS and set `RECOVERED_SPLIT_DIR` below. **Every image ID in
that combined exclusion set must be dropped from v2's training data**, even
though v2 draws from a larger, freshly-combined pool -- otherwise a v1
held-out image could land in v2's training set by chance, and any v1-vs-v2
comparison on it would be silently invalid. `experiments/recoverHeldOutSplit.py`
implements and sanity-checks this same exclusion set on the dev machine; the
cell below reimplements only the load + exclude step (not the sanity checks,
which need local file access this notebook won't have on Kaggle).


In [ ]:
import os

# =====================================================================
#  CONFIG
# =====================================================================
IDRID_ROOT = "/kaggle/input/idrid-disease-grading"
APTOS_ROOT = "/kaggle/input/aptos2019-blindness-detection"

# EyePACS (Kaggle "Diabetic Retinopathy Detection" competition data, or a
# pre-curated/quality-filtered subset of it uploaded as its own dataset --
# either way it must have an image folder + a CSV with image/level columns).
EYEPACS_ROOT = "/kaggle/input/diabetic-retinopathy-detection"
USE_EYEPACS = True  # False -> v2 architecture/loss changes only, no new data
                     # (still useful: isolates items 1/3/4 from items 2/5)

# v1's own recovered held-out split -- upload branchA_v1_{test,val}_{ids,labels}.npy
# as a private Kaggle dataset. MUST be excluded from training; see markdown above.
RECOVERED_SPLIT_DIR = "/kaggle/input/branch-a-v1-recovered-split"

USE_IDRID = True
DEBUG_MAX_PER_SOURCE = None   # e.g. 200 for a fast end-to-end pipeline check

OUT_DIR = "/kaggle/working"
CACHE_DIR = "/kaggle/temp/bg_cache_512"

# ---- item 1: resolution ----
# 512, not 640. Chosen, not defaulted: every other model in this pipeline
# (M2 vessel, M3 localization, M4/M5 lesions) already runs at 512, so this
# keeps one "resolution family" across the whole system rather than adding a
# second, and it is still a large jump from 384 (~1.78x linear, ~3.16x pixel
# count) -- expected to be the largest single lever on the grade-3/4 boundary
# specifically, per the v2 task's own diagnosis (384px too coarse to resolve
# fine neovascular fronds). 640 costs materially more VRAM/time for a
# resolution nothing else here uses; if 512 does not move grade-4 recall
# enough on the recovered held-out set (see the evaluation cell), sweep 640
# next rather than assuming it would have helped.
IMG_SIZE = 512

SEED = 42
BATCH_SIZE = 24          # smaller than v1's 32 -- 512px costs ~1.78x the
                          # activation memory of 384px per image; raise this
                          # if your GPU has headroom (T4/P100: 16GB, should
                          # comfortably fit 24-32 at 512 for EfficientNet-B0)
EPOCHS = 30
PATIENCE = 7
LR = 1e-4
WEIGHT_DECAY = 1e-5
DROP_RATE = 0.3
NUM_WORKERS = 4
USE_AMP = True
MODEL_NAME = "efficientnet_b0"
NUM_CLASSES = 5

# item 4: explicit grade-4-over-grade-3 boost. See the class-weight cell for
# why plain inverse-frequency weighting gets this backwards and what number
# this produces against v1's real counts.
GRADE4_WEIGHT_BOOST = 1.3

# item 5: binary head
REFERABLE_FROM = 2
FOCAL_GAMMA = 2.0
BINARY_LOSS_WEIGHT = 0.5   # relative to the 5-class ordinal loss; both are
                            # logged separately every epoch so this can be
                            # retuned without re-deriving it from scratch
TARGET_SENSITIVITY = 0.90  # item 5's locked operating point

CKPT_PATH         = f"{OUT_DIR}/branchA_v2.pt"
VAL_LOGITS_PATH   = f"{OUT_DIR}/branchA_v2_val_logits.npy"
VAL_LABELS_PATH   = f"{OUT_DIR}/branchA_v2_val_labels.npy"
VAL_IDS_PATH      = f"{OUT_DIR}/branchA_v2_val_ids.npy"
VAL_BIN_LOGITS_PATH = f"{OUT_DIR}/branchA_v2_val_binary_logits.npy"
TEST_LOGITS_PATH  = f"{OUT_DIR}/branchA_v2_test_logits.npy"
TEST_LABELS_PATH  = f"{OUT_DIR}/branchA_v2_test_labels.npy"
TEST_IDS_PATH     = f"{OUT_DIR}/branchA_v2_test_ids.npy"
TEST_BIN_LOGITS_PATH = f"{OUT_DIR}/branchA_v2_test_binary_logits.npy"
METRICS_PATH      = f"{OUT_DIR}/branchA_v2_metrics.json"

print("IDRID_ROOT   :", IDRID_ROOT, "| exists:", os.path.isdir(IDRID_ROOT))
print("APTOS_ROOT   :", APTOS_ROOT, "| exists:", os.path.isdir(APTOS_ROOT))
print("EYEPACS_ROOT :", EYEPACS_ROOT, "| exists:", os.path.isdir(EYEPACS_ROOT), "| USE_EYEPACS:", USE_EYEPACS)
print("RECOVERED_SPLIT_DIR:", RECOVERED_SPLIT_DIR, "| exists:", os.path.isdir(RECOVERED_SPLIT_DIR))


In [ ]:
import os, json, random, time, glob, warnings
from concurrent.futures import ThreadPoolExecutor

import numpy as np
import pandas as pd
import cv2
from PIL import Image
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torch.optim.lr_scheduler import CosineAnnealingLR
import torchvision.transforms as T
import timm

from sklearn.model_selection import train_test_split
from sklearn.metrics import (cohen_kappa_score, f1_score, confusion_matrix,
                             roc_curve, recall_score)

warnings.filterwarnings("ignore", category=UserWarning)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("torch", torch.__version__, "| timm", timm.__version__, "| device:", device)
if device.type == "cuda":
    p = torch.cuda.get_device_properties(0)
    print("GPU:", torch.cuda.get_device_name(0), f"| {p.total_memory / 1e9:.1f} GB")
else:
    print("WARNING: no GPU detected - enable Settings -> Accelerator -> GPU T4")

try:
    from torch.amp import autocast as _ac, GradScaler as _GS
    def amp_autocast():
        return _ac("cuda", enabled=USE_AMP and device.type == "cuda")
    def make_scaler():
        return _GS("cuda", enabled=USE_AMP and device.type == "cuda")
except Exception:
    from torch.cuda.amp import autocast as _ac, GradScaler as _GS
    def amp_autocast():
        return _ac(enabled=USE_AMP and device.type == "cuda")
    def make_scaler():
        return _GS(enabled=USE_AMP and device.type == "cuda")

def seed_everything(seed=SEED):
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

def seed_worker(worker_id):
    s = torch.initial_seed() % (2 ** 32)
    np.random.seed(s)
    random.seed(s)

seed_everything(SEED)
g = torch.Generator()
g.manual_seed(SEED)

os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(CACHE_DIR, exist_ok=True)
print("cache dir :", CACHE_DIR)
print("output dir:", OUT_DIR)


## 1. Ben Graham preprocessing (inlined, UNCHANGED from v1)

Identical to `preprocessing/ben_graham.py` and to v1's own copy of this cell
-- only `target_size` (now 512 by default via `IMG_SIZE`) differs, and that
was already a parameter, not a hardcoded value. Do not "improve" this
function; see `preprocessModel1.m`'s header for why a changed preprocessing
recipe is a silent accuracy regression, not an improvement, once a model has
trained against a specific one.

In [ ]:
def ben_graham_preprocess(image: np.ndarray, target_size: int = 384) -> np.ndarray:
    """Ben Graham-style fundus preprocessing. Identical to preprocessing/ben_graham.py.

    image       : BGR uint8 array, shape (H, W, 3)
    target_size : output square side length
    returns     : uint8 array (target_size, target_size, 3), values 0..255, BGR order
    """
    if image is None or image.size == 0:
        raise ValueError("ben_graham_preprocess received an empty or None image.")
    if image.ndim != 3 or image.shape[2] != 3:
        raise ValueError(f"Expected a 3-channel image, got shape {image.shape}.")

    gray = image[:, :, 1]
    _, mask = cv2.threshold(gray, 7, 255, cv2.THRESH_BINARY)
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (15, 15))
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel)

    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if contours:
        largest = max(contours, key=cv2.contourArea)
        x, y, w, h = cv2.boundingRect(largest)
        x, y = max(0, x), max(0, y)
        w = min(w, image.shape[1] - x)
        h = min(h, image.shape[0] - y)
        cropped = image[y:y + h, x:x + w]
    else:
        cropped = image
    if cropped.size == 0:
        cropped = image

    resized = cv2.resize(cropped, (target_size, target_size), interpolation=cv2.INTER_AREA)

    sigma = target_size / 30.0
    ksize = max(int(sigma) * 2 + 1, 1)
    blurred = cv2.GaussianBlur(resized, (ksize, ksize), sigma)
    enhanced = cv2.addWeighted(resized, 4, blurred, -4, 128)
    return enhanced


_t = np.zeros((512, 512, 3), np.uint8)
cv2.circle(_t, (256, 256), 200, (40, 180, 60), -1)
_o = ben_graham_preprocess(_t, IMG_SIZE)
print("self-test OK - output", _o.shape, _o.dtype, "range", int(_o.min()), int(_o.max()))


## 2. Data loading: APTOS + IDRiD (unchanged) + EyePACS (new, item 2)

EyePACS's own Kaggle "Diabetic Retinopathy Detection" competition labels
(`0`=No DR .. `4`=Proliferative DR) are ALREADY on the same ICDR 0-4 scale
APTOS and IDRiD use -- "ICDR-harmonized" here is mostly verification, not
remapping: `load_eyepacs` asserts the label column only contains `{0..4}`
and fails loudly if it does not, rather than silently coercing something
that turns out not to match.

"Curated/quality-filtered subset": EyePACS is large and famously noisy
(off-center, out-of-focus, or artifact-heavy photos are common in the raw
competition set). `eyepacs_quality_ok` is a lightweight OpenCV filter, not a
port of the project's own MATLAB quality gate (that only runs in MATLAB, and
this notebook has no MATLAB) -- it rejects images that are too dark/bright on
average, too low-contrast (a cheap blur proxy via Laplacian variance), or
implausibly small. Thresholds are marked as a starting point, not fitted
numbers -- re-tune them against a labeled sample of EyePACS images actually
rejected/kept before trusting this at scale, the same way `redFloor`/
`grade3QuadMin` in `ruleEngineGrade.m` were fitted against real held-out
images rather than picked by feel.

In [ ]:
def eyepacs_quality_ok(bgr, min_mean=15, max_mean=240, min_lap_var=15.0, min_side=256):
    """Cheap, disclosed-as-unfitted quality gate for raw EyePACS images.

    Not the project's real quality gate (MATLAB-only, not portable here).
    Rejects: too dark, too bright/washed-out, too blurry (low Laplacian
    variance -- the same focus proxy phc-local-app's assessFocus.m uses,
    just not the same fitted threshold), or too small to be a real fundus
    photo rather than a thumbnail/corrupt file.
    """
    if bgr is None or bgr.size == 0:
        return False
    h, w = bgr.shape[:2]
    if min(h, w) < min_side:
        return False
    gray = cv2.cvtColor(bgr, cv2.COLOR_BGR2GRAY)
    mean_val = float(gray.mean())
    if mean_val < min_mean or mean_val > max_mean:
        return False
    lap_var = float(cv2.Laplacian(gray, cv2.CV_64F).var())
    if lap_var < min_lap_var:
        return False
    return True


def load_aptos(root):
    csv = os.path.join(root, "train.csv")
    img_dir = os.path.join(root, "train_images")
    if not os.path.isfile(csv):
        raise FileNotFoundError(f"APTOS train.csv not found at {csv}.")
    df = pd.read_csv(csv).rename(columns={"id_code": "image_id", "diagnosis": "grade"})
    df["path"] = df["image_id"].apply(lambda s: os.path.join(img_dir, f"{s}.png"))
    df["source"] = "aptos"
    df["grade"] = pd.to_numeric(df["grade"], errors="coerce")
    df = df.dropna(subset=["grade"])
    df["grade"] = df["grade"].astype(int)
    return df[["image_id", "path", "grade", "source"]]


def _tag_of(path):
    low = path.lower()
    return "test" if "test" in low else ("train" if "train" in low else "all")


def load_idrid(root):
    if not os.path.isdir(root):
        raise FileNotFoundError(f"IDRID_ROOT '{root}' does not exist.")
    imgs = []
    for e in ("*.jpg", "*.jpeg", "*.JPG", "*.JPEG", "*.png"):
        imgs += glob.glob(os.path.join(root, "**", e), recursive=True)
    by_tag_stem, by_stem = {}, {}
    for p in imgs:
        stem = os.path.splitext(os.path.basename(p))[0]
        by_tag_stem.setdefault((_tag_of(p), stem), p)
        by_stem.setdefault(stem, p)

    rows = []
    for c in glob.glob(os.path.join(root, "**", "*.csv"), recursive=True):
        try:
            t = pd.read_csv(c)
        except Exception:
            continue
        t.columns = [str(x).strip() for x in t.columns]
        gcol = next((col for col in t.columns
                     if col.lower().replace("_", " ").startswith("retinopathy grade")), None)
        ncol = next((col for col in t.columns
                     if col.lower().replace("_", " ").startswith("image name")), None)
        if not (gcol and ncol):
            continue
        ctag = _tag_of(c)
        t = t[[ncol, gcol]].rename(columns={ncol: "name", gcol: "grade"})
        t["name"] = t["name"].astype(str).str.strip()
        t["grade"] = pd.to_numeric(t["grade"], errors="coerce")
        t = t.dropna(subset=["name", "grade"])
        for name, grade in zip(t["name"], t["grade"]):
            p = by_tag_stem.get((ctag, name)) or by_stem.get(name)
            if p is None:
                continue
            rows.append({"image_id": f"idrid_{ctag}_{name}", "path": p,
                         "grade": int(grade), "source": "idrid"})
    if not rows:
        raise FileNotFoundError(f"No IDRiD grade CSV rows matched under {root}.")
    df = pd.DataFrame(rows).drop_duplicates(subset="image_id").reset_index(drop=True)
    return df[["image_id", "path", "grade", "source"]]


def load_eyepacs(root, quality_filter=True, max_images=None):
    """EyePACS: image/level CSV (trainLabels.csv in the original Kaggle
    competition) + an images folder. quality_filter=True applies
    eyepacs_quality_ok per-image -- this reads every file once (slow: EyePACS
    is ~35k-88k images depending on which split you attach) but there is no
    metadata field to filter on instead; a real quality label was never
    collected for this dataset.
    """
    if not os.path.isdir(root):
        raise FileNotFoundError(
            f"EYEPACS_ROOT '{root}' does not exist. Attach the EyePACS dataset "
            f"(or your own curated subset of it) via Kaggle's Add Data panel.")
    csvs = glob.glob(os.path.join(root, "**", "*.csv"), recursive=True)
    label_csv = next((c for c in csvs if "label" in os.path.basename(c).lower()), None)
    if label_csv is None:
        raise FileNotFoundError(f"No labels CSV found under {root} (expected e.g. trainLabels.csv).")
    t = pd.read_csv(label_csv)
    t.columns = [c.strip().lower() for c in t.columns]
    img_col = next((c for c in t.columns if c in ("image", "id_code", "image_id")), t.columns[0])
    lvl_col = next((c for c in t.columns if c in ("level", "diagnosis", "grade")), t.columns[1])
    t = t.rename(columns={img_col: "image_id", lvl_col: "grade"})
    t["grade"] = pd.to_numeric(t["grade"], errors="coerce")
    t = t.dropna(subset=["grade"])
    t["grade"] = t["grade"].astype(int)
    # item 2: ICDR harmonization is a VERIFICATION, not a remap -- EyePACS
    # already uses 0-4. Fail loudly rather than silently coercing an
    # unexpected label set into range.
    bad = set(t["grade"].unique()) - {0, 1, 2, 3, 4}
    assert not bad, f"EyePACS grade column has out-of-ICDR-range values: {bad} -- verify the label mapping before proceeding."

    img_files = {}
    for e in ("*.jpg", "*.jpeg", "*.png"):
        for p in glob.glob(os.path.join(root, "**", e), recursive=True):
            img_files[os.path.splitext(os.path.basename(p))[0]] = p
    t["path"] = t["image_id"].astype(str).map(img_files)
    t = t.dropna(subset=["path"]).reset_index(drop=True)

    if max_images:
        t = t.sample(min(len(t), max_images), random_state=SEED).reset_index(drop=True)

    if quality_filter:
        keep = []
        for p in tqdm(t["path"], desc="EyePACS quality filter", mininterval=5.0):
            img = cv2.imread(p)
            keep.append(eyepacs_quality_ok(img))
        n_before = len(t)
        t = t[np.array(keep)].reset_index(drop=True)
        print(f"EyePACS quality filter: kept {len(t)}/{n_before} "
              f"({100*len(t)/max(n_before,1):.1f}%) -- thresholds are a documented "
              f"starting point (see markdown above), not fitted.")

    t["source"] = "eyepacs"
    return t[["image_id", "path", "grade", "source"]]


frames = [load_aptos(APTOS_ROOT)]
if USE_IDRID:
    frames.append(load_idrid(IDRID_ROOT))
if USE_EYEPACS:
    frames.append(load_eyepacs(EYEPACS_ROOT))

data = pd.concat(frames, ignore_index=True)
data = data[data["grade"].between(0, 4)].reset_index(drop=True)
data["image_id"] = data["source"] + "__" + data["image_id"].astype(str)

exists = data["path"].apply(os.path.isfile)
if (~exists).any():
    print(f"WARNING: {(~exists).sum()} rows point to a missing image file - dropped")
data = data[exists].reset_index(drop=True)

if DEBUG_MAX_PER_SOURCE:
    data = pd.concat([
        d.sample(min(len(d), DEBUG_MAX_PER_SOURCE), random_state=SEED)
        for _, d in data.groupby("source")
    ]).reset_index(drop=True)
    print(f"DEBUG_MAX_PER_SOURCE={DEBUG_MAX_PER_SOURCE} -> {len(data)} images")

print("\nTotal usable images (pre held-out exclusion):", len(data))
print(pd.crosstab(data["source"], data["grade"], margins=True))


## 3. Exclude v1's recovered held-out split (item 7), THEN stratified split

Two separate steps, in this order on purpose: excluding first means the
70/15/15 split below never had a chance to put a v1-held-out image into
training even transiently. `image_id` here is already `<source>__<id>`
(built above), matching `branchA_v1_{test,val}_ids.npy`'s own id format
exactly -- see `experiments/recoverHeldOutSplit.py` for how those were
produced and verified.

In [ ]:
recovered_test_ids = np.load(os.path.join(RECOVERED_SPLIT_DIR, "branchA_v1_test_ids.npy"), allow_pickle=True)
recovered_val_ids = np.load(os.path.join(RECOVERED_SPLIT_DIR, "branchA_v1_val_ids.npy"), allow_pickle=True)
excluded_ids = set(recovered_test_ids) | set(recovered_val_ids)
print(f"recovered v1 held-out ids to exclude from v2 training: {len(excluded_ids)}")

n_before = len(data)
data_trainable = data[~data["image_id"].isin(excluded_ids)].reset_index(drop=True)
n_excluded_present = n_before - len(data_trainable)
print(f"excluded {n_excluded_present} images that were in this combined pool AND in v1's held-out set")
print(f"remaining pool for v2 train/val/test split: {len(data_trainable)}")

# aptos ids in the recovered split are content hashes (e.g. '89ee1fa16f90'),
# not filenames, so n_excluded_present will UNDER-count aptos exclusions
# unless this notebook's aptos image_id matches that same hash scheme. If
# your APTOS loader produces plain competition ids instead, generate the
# same hash here before comparing, or you will silently under-exclude aptos
# held-out images. Verify this count against len(excluded_ids) restricted to
# idrid ids (directly comparable) as a sanity floor:
idrid_excluded_expected = sum(1 for i in excluded_ids if i.startswith("idrid"))
idrid_excluded_actual = sum(1 for i in data["image_id"] if i in excluded_ids and i.startswith("idrid"))
print(f"idrid-only exclusion sanity check: {idrid_excluded_actual} matched "
      f"(expect up to {idrid_excluded_expected}, exact figure depends on how "
      f"much of IDRiD is attached here)")
assert idrid_excluded_actual > 0 or idrid_excluded_expected == 0, (
    "expected at least some idrid held-out ids to match and exclude -- if this "
    "fires, the id-construction scheme above does not match "
    "recoverHeldOutSplit.py's, and APTOS exclusion is probably also silently "
    "broken. Fix the id scheme before training, not after.")


def split_70_15_15(df, seed=SEED):
    idx = np.arange(len(df))
    y = df["grade"].values
    tr, tmp = train_test_split(idx, test_size=0.30, random_state=seed, stratify=y)
    va, te = train_test_split(tmp, test_size=0.50, random_state=seed, stratify=y[tmp])
    return df.iloc[tr], df.iloc[va], df.iloc[te]


parts = {"train": [], "val": [], "test": []}
for src, d in data_trainable.groupby("source"):
    d = d.reset_index(drop=True)
    tr, va, te = split_70_15_15(d, SEED)
    parts["train"].append(tr)
    parts["val"].append(va)
    parts["test"].append(te)
    print(f"{src:8s} -> train {len(tr):5d} | val {len(va):4d} | test {len(te):4d}")

train_df = pd.concat(parts["train"]).sample(frac=1, random_state=SEED).reset_index(drop=True)
val_df = pd.concat(parts["val"]).reset_index(drop=True)
test_df = pd.concat(parts["test"]).reset_index(drop=True)

a, b, c = set(train_df.image_id), set(val_df.image_id), set(test_df.image_id)
assert not (a & b) and not (a & c) and not (b & c), "split leakage detected"
assert not (a & excluded_ids), "a recovered held-out id leaked into v2 TRAINING -- stop and fix before spending compute"

print(f"\nCOMBINED  train {len(train_df)} | val {len(val_df)} | test {len(test_df)}")
print("\ntrain grade x source:")
print(pd.crosstab(train_df.source, train_df.grade, margins=True))


## 4. Ben Graham preprocess + cache to disk (unchanged pattern, new resolution)

In [ ]:
def cache_key(source, image_id):
    return f"{source}_{image_id}_{IMG_SIZE}"


def cache_path_for(source, image_id):
    return os.path.join(CACHE_DIR, cache_key(source, image_id) + ".npy")


def _preprocess_and_cache(row):
    out_path = cache_path_for(row.source, row.image_id)
    if os.path.isfile(out_path):
        return True
    bgr = cv2.imread(row.path, cv2.IMREAD_COLOR)
    if bgr is None:
        return False
    proc = ben_graham_preprocess(bgr, IMG_SIZE)
    rgb = cv2.cvtColor(proc, cv2.COLOR_BGR2RGB)
    np.save(out_path, rgb)
    return True


all_rows = pd.concat([train_df, val_df, test_df], ignore_index=True)
with ThreadPoolExecutor(max_workers=8) as ex:
    results = list(tqdm(ex.map(_preprocess_and_cache, [r for _, r in all_rows.iterrows()]),
                        total=len(all_rows), desc="ben_graham cache", mininterval=5.0))
n_failed = sum(1 for ok in results if not ok)
print(f"cached {len(results) - n_failed}/{len(results)} images at {IMG_SIZE}x{IMG_SIZE} ({n_failed} failed to read)")


## 5. Dataset + augmentation (unchanged from v1 except IMG_SIZE, and the
binary referable label added alongside the 5-class one)

In [ ]:
IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)

train_tf = T.Compose([
    T.RandomResizedCrop(IMG_SIZE, scale=(0.8, 1.0), ratio=(0.9, 1.1111)),
    T.RandomRotation(20, fill=0),
    T.RandomHorizontalFlip(0.5),
    T.ColorJitter(brightness=0.15, contrast=0.15),
    T.ToTensor(),
    T.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])
eval_tf = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE)),
    T.ToTensor(),
    T.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])


class RetinaDataset(Dataset):
    def __init__(self, df, transform):
        self.df = df.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, i):
        r = self.df.iloc[i]
        arr = np.load(cache_path_for(r.source, r.image_id))
        img = self.transform(Image.fromarray(arr))
        grade = int(r.grade)
        referable = float(grade >= REFERABLE_FROM)
        return img, grade, referable, cache_key(r.source, r.image_id)


train_ds = RetinaDataset(train_df, train_tf)
val_ds = RetinaDataset(val_df, eval_tf)
test_ds = RetinaDataset(test_df, eval_tf)

# item 5: oversample referable EyePACS cases specifically. EyePACS's own
# class imbalance is more severe than APTOS/IDRiD's (heavily grade-0
# dominant), and it is the largest source in the combined pool once
# attached, so weighting the SAMPLER (not just the loss) keeps referable
# cases from being drowned out epoch-to-epoch, not just down-weighted in a
# loss term that still sees them rarely.
sample_weights = np.ones(len(train_df), dtype=np.float64)
is_eyepacs_referable = (train_df["source"].values == "eyepacs") & (train_df["grade"].values >= REFERABLE_FROM)
if is_eyepacs_referable.any():
    referable_count = int(is_eyepacs_referable.sum())
    nonreferable_count = len(train_df) - referable_count
    boost = nonreferable_count / max(referable_count, 1)
    sample_weights[is_eyepacs_referable] = boost
    print(f"EyePACS referable oversampling: {referable_count} referable cases "
          f"boosted {boost:.2f}x in the training sampler")
sampler = WeightedRandomSampler(sample_weights, num_samples=len(train_df), replacement=True)

loader_kw = dict(batch_size=BATCH_SIZE, num_workers=NUM_WORKERS, pin_memory=True,
                 worker_init_fn=seed_worker, persistent_workers=NUM_WORKERS > 0)
train_loader = DataLoader(train_ds, sampler=sampler, drop_last=True, generator=g, **loader_kw)
val_loader = DataLoader(val_ds, shuffle=False, **loader_kw)
test_loader = DataLoader(test_ds, shuffle=False, **loader_kw)

xb, yb, rb, idb = next(iter(train_loader))
print("sample batch:", tuple(xb.shape), xb.dtype, "| grades", yb[:8].tolist(), "| referable", rb[:8].tolist())


## 6. Model - EfficientNet-B0, TWO heads (item 2)

Same backbone family as v1 (`timm` EfficientNet-B0, ImageNet-pretrained),
retrained at 512px. `headBin` is new: a second linear layer reading the SAME
pooled+dropout features as the 5-class head, trained jointly. Sharing the
trunk (rather than a fully separate binary model) is deliberate -- both
heads are answering questions about the same underlying pathology signal,
and a shared trunk means the binary head benefits from EyePACS's larger
referable-case volume even though the 5-class head is still evaluated
primarily on APTOS+IDRiD's more reliable ICDR grading.

In [ ]:
class DRClassifierV2(nn.Module):
    """timm backbone (feature mode) + explicit Dropout + TWO linear heads.

    self.head5   - 5-class ICDR grade (unchanged role from v1's DRClassifier)
    self.headBin - binary referable/non-referable (item 2), one logit,
                   BCEWithLogitsLoss-compatible.

    The explicit nn.Dropout before both heads is still what MC-Dropout
    (Task 6.1) toggles at inference -- unchanged from v1, and load-bearing
    for the SAME reason: timm's own head dropout is functional, not a
    module, and MC-Dropout needs a module to force into train() mode.
    """
    def __init__(self, model_name, num_classes, drop_rate, pretrained=True):
        super().__init__()
        self.backbone = timm.create_model(model_name, pretrained=pretrained,
                                          num_classes=0, drop_rate=0.0)
        self.num_features = self.backbone.num_features
        self.drop = nn.Dropout(p=drop_rate)
        self.head5 = nn.Linear(self.num_features, num_classes)
        self.headBin = nn.Linear(self.num_features, 1)

    def forward(self, x):
        feats = self.drop(self.backbone(x))
        return self.head5(feats), self.headBin(feats).squeeze(-1)


ARCH_DESC = ("DRClassifierV2: timm(model_name, num_classes=0, drop_rate=0) -> "
             "nn.Dropout(drop_rate) -> {head5: Linear(num_features,5), "
             "headBin: Linear(num_features,1)}")

try:
    model = DRClassifierV2(MODEL_NAME, NUM_CLASSES, DROP_RATE, pretrained=True).to(device)
except Exception as e:
    raise RuntimeError(
        "Could not create the pretrained model. Turn Settings -> Internet -> On so timm "
        "can download ImageNet weights, then re-run this cell.") from e

n_params = sum(p.numel() for p in model.parameters()) / 1e6
n_drop = sum(1 for m in model.modules() if isinstance(m, nn.Dropout) and m.p > 0)
print(f"{MODEL_NAME} (dual head): {n_params:.1f}M params | dropout modules: {n_drop} | img_size {IMG_SIZE}")
assert n_drop >= 1, "expected a real nn.Dropout before the heads for MC-Dropout"


## 7. Losses (items 3, 4, 5)

Three pieces, reported separately every epoch so a regression in one doesn't
hide inside a single combined number:

1. **Class weights (item 4).** Inverse-frequency weighting alone gets grade
   4 vs grade 3 backwards whenever grade 4 happens to have MORE raw training
   examples than grade 3 (true for v1: 250 vs 200) -- the formula only
   compensates for imbalance, it has no idea grade 4 is clinically worse to
   miss. `GRADE4_WEIGHT_BOOST` is applied ON TOP of inverse-frequency
   weighting, specifically and only to grade 4, and the cell asserts the
   result actually reversed the ordering before training starts.

2. **`OrdinalWeightedCEv2` (item 3).** v1's ordinal factor was
   `1 + |pred - target|` -- symmetric, so over- and under-grading by the same
   amount cost the same. v2's is asymmetric: `1 + (true-pred)^2` when the
   model UNDER-grades (pred < true), `1 + 0.5*(true-pred)^2` when it
   OVER-grades or is exact. Squared rather than linear specifically to
   penalize LARGE under-grading errors (grade 0 predicted for a true grade 4)
   much more than small ones (grade 3 predicted for a true grade 4) -- this
   is the loss-side lever aimed at recovering grade-4 recall, aligned with
   QWK's own quadratic penalty instead of leaving CE and the selection metric
   mismatched (the v2 task's own framing).

3. **`FocalLossBCE` (item 5).** Standard focal loss on the binary head,
   `gamma=FOCAL_GAMMA`, combined with the class-balanced sampler already
   built into `train_loader` above (oversampling handles the volume
   imbalance; focal loss handles the easy-negative-dominance imbalance within
   each batch -- the two address different things and neither substitutes for
   the other).

In [ ]:
counts = (train_df["grade"].value_counts()
          .reindex(range(NUM_CLASSES), fill_value=0).sort_index())
total = int(counts.sum())
w_raw = total / (NUM_CLASSES * counts.clip(lower=1).astype(float))

w_boosted = w_raw.copy()
w_boosted[4] = w_raw[4] * GRADE4_WEIGHT_BOOST

print("train grade counts     :", counts.to_dict())
print("inverse-freq weights   :", {k: round(v, 3) for k, v in w_raw.to_dict().items()})
print(f"grade4-boosted weights :", {k: round(v, 3) for k, v in w_boosted.to_dict().items()})
print(f"grade3 weight {w_raw[3]:.3f} vs grade4 weight {w_boosted[4]:.3f} "
      f"(boost x{GRADE4_WEIGHT_BOOST}) -> grade4 > grade3: {w_boosted[4] > w_raw[3]}")
assert w_boosted[4] > w_raw[3], (
    "GRADE4_WEIGHT_BOOST is not large enough to reverse the v1 ordering against "
    "THIS run's actual class counts -- raise it and re-check before training. "
    "Do not train with grade 4 weighted below grade 3; that was the bug being fixed.")

class_weights = torch.tensor(w_boosted.values, dtype=torch.float32, device=device)


class OrdinalWeightedCEv2(nn.Module):
    """Inverse-freq + grade4-boosted weighted CE, scaled per-sample by an
    ASYMMETRIC ordinal factor -- see markdown above for the exact formula
    and why it is squared and asymmetric."""
    def __init__(self, class_weights):
        super().__init__()
        self.register_buffer("cw", class_weights.detach().clone())

    def forward(self, logits, target):
        ce = F.cross_entropy(logits, target, weight=self.cw, reduction="none")
        pred = logits.detach().argmax(dim=1)
        diff = (target - pred).float()          # true - pred, signed
        under = diff > 0                         # predicted LOWER than true
        penalty = torch.where(under, diff.pow(2), 0.5 * diff.pow(2))
        factor = 1.0 + penalty
        return (ce * factor).mean()


class FocalLossBCE(nn.Module):
    """Binary focal loss on raw logits (numerically stable via
    binary_cross_entropy_with_logits, not sigmoid then BCE)."""
    def __init__(self, gamma=2.0, pos_weight=None):
        super().__init__()
        self.gamma = gamma
        self.pos_weight = pos_weight

    def forward(self, logits, target):
        bce = F.binary_cross_entropy_with_logits(
            logits, target, reduction="none", pos_weight=self.pos_weight)
        p = torch.sigmoid(logits)
        p_t = p * target + (1 - p) * (1 - target)
        focal = bce * (1 - p_t).pow(self.gamma)
        return focal.mean()


criterion5 = OrdinalWeightedCEv2(class_weights).to(device)
criterionBin = FocalLossBCE(gamma=FOCAL_GAMMA).to(device)
print(f"\nlosses ready: OrdinalWeightedCEv2 (5-class) + FocalLossBCE gamma={FOCAL_GAMMA} (binary)")
print(f"combined as: {1.0} * ordinal_loss + {BINARY_LOSS_WEIGHT} * focal_loss")


## 8. Metrics (item 7: ALL FOUR, every epoch and at final evaluation)

QWK alone is what the v2 task explicitly warns against checking in
isolation: it is insensitive to 54 grade-4 images inside a 628-image test
set and would hide a grade-4-recall regression completely. `compute_metrics`
below always returns all four: QWK, referable sensitivity/specificity,
grade-4 recall, grade-1 recall -- and the training loop and final evaluation
both print all four, not a subset.

In [ ]:
def compute_metrics(y_true, y_pred, referable_from=REFERABLE_FROM):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    qwk = cohen_kappa_score(y_true, y_pred, weights="quadratic")
    macro_f1 = f1_score(y_true, y_pred, average="macro", labels=list(range(NUM_CLASSES)))

    ref_true = y_true >= referable_from
    ref_pred = y_pred >= referable_from
    tp = int((ref_true & ref_pred).sum())
    fn = int((ref_true & ~ref_pred).sum())
    tn = int((~ref_true & ~ref_pred).sum())
    fp = int((~ref_true & ref_pred).sum())
    ref_sens = tp / max(tp + fn, 1)
    ref_spec = tn / max(tn + fp, 1)

    g4_mask = y_true == 4
    g4_recall = recall_score(y_true[g4_mask], y_pred[g4_mask], labels=[4], average="micro") if g4_mask.any() else float("nan")
    g1_mask = y_true == 1
    g1_recall = recall_score(y_true[g1_mask], y_pred[g1_mask], labels=[1], average="micro") if g1_mask.any() else float("nan")

    mean_signed_error = float((y_pred - y_true).mean())  # negative = systematic under-grading

    return {
        "qwk": float(qwk), "macro_f1": float(macro_f1),
        "ref_sens": float(ref_sens), "ref_spec": float(ref_spec),
        "grade4_recall": float(g4_recall), "grade4_n": int(g4_mask.sum()),
        "grade1_recall": float(g1_recall), "grade1_n": int(g1_mask.sum()),
        "mean_signed_error": mean_signed_error,
    }


@torch.no_grad()
def evaluate(model, loader, criterion5, criterionBin):
    model.eval()
    all_logits5, all_logits_bin, all_labels, all_ref, all_ids = [], [], [], [], []
    run_loss5, run_lossBin, seen = 0.0, 0.0, 0
    for imgs, labels, referable, ids in loader:
        imgs = imgs.to(device, non_blocking=True)
        labels_t = labels.to(device, non_blocking=True)
        ref_t = referable.to(device, non_blocking=True).float()
        with amp_autocast():
            logits5, logitsBin = model(imgs)
            l5 = criterion5(logits5, labels_t)
            lb = criterionBin(logitsBin, ref_t)
        run_loss5 += l5.item() * imgs.size(0)
        run_lossBin += lb.item() * imgs.size(0)
        seen += imgs.size(0)
        all_logits5.append(logits5.float().cpu().numpy())
        all_logits_bin.append(logitsBin.float().cpu().numpy())
        all_labels.append(labels.numpy())
        all_ref.append(referable.numpy())
        all_ids.extend(ids)

    logits5 = np.concatenate(all_logits5)
    logits_bin = np.concatenate(all_logits_bin)
    labels = np.concatenate(all_labels)
    y_pred = logits5.argmax(axis=1)
    m = compute_metrics(labels, y_pred)
    m["loss5"] = run_loss5 / max(seen, 1)
    m["lossBin"] = run_lossBin / max(seen, 1)
    m["loss"] = m["loss5"] + BINARY_LOSS_WEIGHT * m["lossBin"]
    return m, logits5, logits_bin, labels, np.array(all_ids)


## 9. Train

In [ ]:
seed_everything(SEED)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = CosineAnnealingLR(optimizer, T_max=EPOCHS)
scaler = make_scaler()

best_qwk, best_epoch, no_improve = -1.0, -1, 0
history = []

print(f"train {len(train_df)} | val {len(val_df)} | batch {BATCH_SIZE} | img {IMG_SIZE} | "
      f"up to {EPOCHS} epochs | early stop patience {PATIENCE}\n")

for epoch in range(1, EPOCHS + 1):
    model.train()
    run_loss, seen, t0 = 0.0, 0, time.time()
    pbar = tqdm(train_loader, desc=f"epoch {epoch:02d}/{EPOCHS} [train]", leave=False, mininterval=5.0)
    for imgs, labels, referable, _ in pbar:
        imgs = imgs.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)
        referable = referable.to(device, non_blocking=True).float()
        optimizer.zero_grad(set_to_none=True)
        with amp_autocast():
            logits5, logitsBin = model(imgs)
            loss5 = criterion5(logits5, labels)
            lossBin = criterionBin(logitsBin, referable)
            loss = loss5 + BINARY_LOSS_WEIGHT * lossBin
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        run_loss += loss.item() * imgs.size(0)
        seen += imgs.size(0)
        pbar.set_postfix(loss=f"{run_loss / seen:.4f}")
    scheduler.step()
    train_loss = run_loss / max(seen, 1)

    vm, v_logits5, v_logits_bin, v_labels, v_ids = evaluate(model, val_loader, criterion5, criterionBin)
    lr_now = optimizer.param_groups[0]["lr"]
    dt = time.time() - t0
    improved = vm["qwk"] > best_qwk + 1e-5
    flag = "   <-- new best" if improved else ""

    # ALL FOUR metrics printed every epoch (item 7) -- not just QWK, so a
    # grade-4/grade-1 regression is visible during training, not discovered
    # only at the end.
    print(f"epoch {epoch:02d}/{EPOCHS} | {dt:5.0f}s | lr {lr_now:.2e} | "
          f"train_loss {train_loss:.4f} | val_loss {vm['loss']:.4f} | "
          f"QWK {vm['qwk']:.4f} | refDR sens {vm['ref_sens']:.3f} spec {vm['ref_spec']:.3f} | "
          f"grade4_recall {vm['grade4_recall']:.3f} (n={vm['grade4_n']}) | "
          f"grade1_recall {vm['grade1_recall']:.3f} (n={vm['grade1_n']}){flag}")

    history.append({"epoch": epoch, "train_loss": float(train_loss), "lr": float(lr_now),
                    "seconds": float(dt), **vm})

    if improved:
        best_qwk, best_epoch, no_improve = vm["qwk"], epoch, 0
        torch.save({
            "model_state_dict": model.state_dict(),
            "model_name": MODEL_NAME,
            "arch": ARCH_DESC,
            "num_classes": NUM_CLASSES,
            "num_features": int(model.num_features),
            "drop_rate": DROP_RATE,
            "img_size": IMG_SIZE,   # <-- 512, not 384. This is the field
                                     # branchAInfer.py / branchAInferMatlab.m's
                                     # calibration version-guard checks.
            "normalize_mean": list(IMAGENET_MEAN),
            "normalize_std": list(IMAGENET_STD),
            "channel_order": "RGB",
            "preprocessing": "ben_graham: circular crop -> resize -> gaussian-subtraction contrast",
            "class_weights": [float(x) for x in class_weights.detach().cpu().tolist()],
            "grade4_weight_boost": GRADE4_WEIGHT_BOOST,
            "train_grade_counts": {int(k): int(v) for k, v in counts.to_dict().items()},
            "referable_from": REFERABLE_FROM,
            "binary_loss_weight": BINARY_LOSS_WEIGHT,
            "focal_gamma": FOCAL_GAMMA,
            "epoch": epoch,
            "val_qwk": float(best_qwk),
            "val_metrics": vm,
            "seed": SEED,
        }, CKPT_PATH)
        np.save(VAL_LOGITS_PATH, v_logits5)
        np.save(VAL_LABELS_PATH, v_labels)
        np.save(VAL_IDS_PATH, v_ids)
        np.save(VAL_BIN_LOGITS_PATH, v_logits_bin)
        print(f"           saved {os.path.basename(CKPT_PATH)} + val logits {tuple(v_logits5.shape)}")
    else:
        no_improve += 1
        if no_improve >= PATIENCE:
            print(f"\nearly stop - no val QWK gain for {PATIENCE} epochs "
                  f"(best epoch {best_epoch}, QWK {best_qwk:.4f})")
            break

print(f"\nbest epoch {best_epoch} | val QWK {best_qwk:.4f}")


## 10. Lock the binary-head threshold on VALIDATION (item 5)

Sweep the ROC curve on the validation split's binary-head logits from the
BEST checkpoint, take the first threshold whose sensitivity is `>=
TARGET_SENSITIVITY` (0.90), and report the specificity it actually achieves
at that point -- honestly, whatever it turns out to be. This is a validation-
only decision; the locked threshold is then APPLIED (not re-swept) on the
test split in the next cell, so the reported test specificity is a real
held-out number, not a re-optimized one.

In [ ]:
ckpt = torch.load(CKPT_PATH, map_location="cpu", weights_only=False)
model.load_state_dict(ckpt["model_state_dict"])
model.to(device)

_, _, val_logits_bin, val_labels, _ = evaluate(model, val_loader, criterion5, criterionBin)
val_probs_bin = 1.0 / (1.0 + np.exp(-val_logits_bin))
val_ref_true = (val_labels >= REFERABLE_FROM).astype(int)

fpr, tpr, thresholds = roc_curve(val_ref_true, val_probs_bin)
ok = tpr >= TARGET_SENSITIVITY
if not ok.any():
    raise RuntimeError(
        f"No threshold on the validation ROC reaches {TARGET_SENSITIVITY:.0%} sensitivity "
        f"(max achieved: {tpr.max():.3f}). Do not lower TARGET_SENSITIVITY to make this pass -- "
        f"report the real max and treat it as a v2 training failure to fix (more epochs, "
        f"more referable oversampling, a different BINARY_LOSS_WEIGHT), not a threshold to relax.")
# roc_curve returns thresholds in DECREASING order; take the LAST index
# where sensitivity still clears the bar -- i.e. the HIGHEST threshold
# (most specific) that still meets the sensitivity floor, not just any
# point that clears it.
idx = np.where(ok)[0][-1]
LOCKED_THRESHOLD = float(thresholds[idx])
achieved_sens = float(tpr[idx])
achieved_spec = float(1 - fpr[idx])

print(f"locked binary threshold: {LOCKED_THRESHOLD:.4f}")
print(f"validation sensitivity at lock: {achieved_sens:.4f} (target >= {TARGET_SENSITIVITY:.0%})")
print(f"validation specificity at lock: {achieved_spec:.4f}  <- report this honestly, do not cherry-pick a different point")


## 11. Held-out test evaluation (item 7: all four metrics, on the
RECOVERED split, threshold applied not re-swept)

In [ ]:
tm, t_logits5, t_logits_bin, t_labels, t_ids = evaluate(model, test_loader, criterion5, criterionBin)

t_probs_bin = 1.0 / (1.0 + np.exp(-t_logits_bin))
t_pred_bin = (t_probs_bin >= LOCKED_THRESHOLD).astype(int)
t_ref_true = (t_labels >= REFERABLE_FROM).astype(int)
tp = int(((t_ref_true == 1) & (t_pred_bin == 1)).sum())
fn = int(((t_ref_true == 1) & (t_pred_bin == 0)).sum())
tn = int(((t_ref_true == 0) & (t_pred_bin == 0)).sum())
fp = int(((t_ref_true == 0) & (t_pred_bin == 1)).sum())
binary_head_sens = tp / max(tp + fn, 1)
binary_head_spec = tn / max(tn + fp, 1)

y_pred5 = t_logits5.argmax(axis=1)
cm = confusion_matrix(t_labels, y_pred5, labels=list(range(NUM_CLASSES)))

print("=" * 70)
print(f"v2 TEST RESULTS (recovered held-out split, n={len(t_labels)})")
print("=" * 70)
print(f"QWK                        : {tm['qwk']:.4f}")
print(f"5-class head referable sens: {tm['ref_sens']:.4f}   spec: {tm['ref_spec']:.4f}")
print(f"binary head referable sens : {binary_head_sens:.4f}   spec: {binary_head_spec:.4f}   "
      f"(locked threshold {LOCKED_THRESHOLD:.4f})")
print(f"grade-4 recall             : {tm['grade4_recall']:.4f}  (n={tm['grade4_n']})")
print(f"grade-1 recall             : {tm['grade1_recall']:.4f}  (n={tm['grade1_n']})")
print(f"mean signed error          : {tm['mean_signed_error']:+.4f}  (negative = under-grading)")
print("\nconfusion matrix (rows=truth, cols=pred):")
print(pd.DataFrame(cm, index=[f"true{i}" for i in range(5)], columns=[f"pred{i}" for i in range(5)]))

# v1 comparison -- paste v1's numbers here from models/evaluation_branchA_v1.txt
# so every v2 run states the delta explicitly rather than leaving the reader
# to go find v1's report and compare by hand.
V1_REFERENCE = {
    "qwk": 0.8688, "ref_sens": 0.8603, "ref_spec": 0.9382,
    "grade4_recall": 0.444, "grade1_recall": 0.000,  # 0/4 on IDRiD specifically
}
print("\nv1 -> v2 delta:")
for k, v1 in V1_REFERENCE.items():
    v2 = tm.get(k, binary_head_sens if k == "ref_sens" else tm.get(k))
    print(f"  {k:15s} v1={v1:.4f}  v2={tm.get(k, float('nan')):.4f}  delta={tm.get(k, float('nan'))-v1:+.4f}")

metrics_out = {
    "test": tm, "val_at_lock": {"sensitivity": achieved_sens, "specificity": achieved_spec},
    "test_binary_head": {"sensitivity": binary_head_sens, "specificity": binary_head_spec,
                         "locked_threshold": LOCKED_THRESHOLD},
    "confusion_matrix": cm.tolist(),
    "v1_reference": V1_REFERENCE,
    "img_size": IMG_SIZE, "grade4_weight_boost": GRADE4_WEIGHT_BOOST,
    "best_epoch": best_epoch, "n_train": len(train_df), "n_val": len(val_df), "n_test": len(test_df),
}
with open(METRICS_PATH, "w") as f:
    json.dump(metrics_out, f, indent=2, default=float)

np.save(TEST_LOGITS_PATH, t_logits5)
np.save(TEST_LABELS_PATH, t_labels)
np.save(TEST_IDS_PATH, t_ids)
np.save(TEST_BIN_LOGITS_PATH, t_logits_bin)
print(f"\nsaved {METRICS_PATH}")


## 12. Artifacts to download -- AND THE MANDATORY NEXT STEP (item 8)

Download every file below from the Kaggle "Output" panel. Then, **before
this model touches anything downstream of a raw checkpoint**:

1. Update `preprocessModel1.m`, `diagnostics/MODEL_INTERFACE_REFERENCE.md`,
   and `docs/model-handoff-guide.md` to say **512** (or whatever `IMG_SIZE`
   actually was for the run you are shipping), not 384. Do this from the
   checkpoint's own `img_size` field, not from memory -- that is exactly how
   the 384-vs-512(claimed as 512 in a stale doc, actually 384) confusion
   happened the first time.
2. Re-run `exportModel1Predictions.py` equivalent against v2's saved val/test
   logits (already in the right shape/format above) and then
   **`calibrateBranchA.m` with `opts.imgSize` set to v2's real resolution**
   -- required, no default, will error otherwise (see that file's header).
   This OVERWRITES `calibration_v1.json`'s `qhat=0.8432`, which was fitted
   for v1 at 384px and means nothing for this model.
3. Confirm the overwrite worked: `branchAInfer.py`'s `load_calibration(ckpt)`
   and `branchAInferMatlab.m` both carry a version guard added specifically
   for this (2026-09-19) that REFUSES to apply a calibration file whose
   `trainedImgSize` doesn't match the loaded model -- if you skip step 2, the
   guard will correctly report the model as UNCALIBRATED rather than
   silently using v1's qhat. Do not treat that warning as a bug to route
   around; it is the safety net this exact mistake needs.

In [ ]:
print("Download from the Kaggle 'Output' panel after the run:\n")
for p in [CKPT_PATH, VAL_LOGITS_PATH, VAL_LABELS_PATH, VAL_IDS_PATH, VAL_BIN_LOGITS_PATH,
          TEST_LOGITS_PATH, TEST_LABELS_PATH, TEST_IDS_PATH, TEST_BIN_LOGITS_PATH, METRICS_PATH]:
    print(" ", p)
print("\nThen: update preprocessModel1.m / MODEL_INTERFACE_REFERENCE.md / model-handoff-guide.md")
print("      to IMG_SIZE, and re-run calibrateBranchA.m with opts.imgSize=IMG_SIZE.")
print("      calibration_v1.json's qhat=0.8432 is NOT valid for this model until then.")
